# Build a simple LLM Application with LCEL:

In [1]:
# This application will translate text from English into another language. It is just a single LLM Call plus some prompting. LCEL is langchain expression language for chaining components together.

In [2]:
%pip install langchain

Note: you may need to restart the kernel to use updated packages.


In [4]:
### Groq: it has introduced LPU Inference Engine which is a hardware and software platform that delivers exceptional compute speed quality
# and energy effeciency. It provides fastest inference for computationally intensive applications with sequential components such as AI language applications like LLMs.


In [40]:
# api keys are in api_all.txt

In [7]:
%pip install huggingface_hub python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Calling open source model:
import os
from dotenv import load_dotenv

from huggingface_hub import login
load_dotenv()

HF_token=os.getenv("HF_token")
login(token=HF_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [9]:
groq_api_key=os.getenv("GROQ_API_KEY")
groq_api_key

'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'

In [10]:
%pip install langchain_groq


   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 2/2 [langchain_groq]

Note: you may need to restart the kernel to use updated packages.


In [12]:
%pip install langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [18]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEndpoint

model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key,temperature=0)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021327AB6750>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021327B42550>, model_name='openai/gpt-oss-20b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [16]:
%pip install langchain_core

Note: you may need to restart the kernel to use updated packages.


In [20]:
from langchain_core.messages import HumanMessage,SystemMessage
messages=[
    SystemMessage(content="translate the following from English to French"),
    HumanMessage(content="hello How are you?")
]

response=model.invoke(messages)                 

In [21]:
response

AIMessage(content='Bonjour, comment vas‑tu\u202f?', additional_kwargs={'reasoning_content': 'We need to translate "hello How are you?" from English to French. The user says "hello How are you?" The translation: "Bonjour, comment ça va ?" or "Bonjour, comment vas-tu ?" The user didn\'t specify formal or informal. Probably informal. So "Bonjour, comment vas-tu ?" Or "Bonjour, comment allez-vous ?" But "hello" is "bonjour". "How are you?" is "Comment vas-tu?" or "Comment allez-vous?" I\'d choose "Bonjour, comment vas-tu ?" It\'s a direct translation. The user wrote "hello How are you?" with capital H in How. So we can respond with "Bonjour, comment vas-tu ?" That is correct.'}, response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 86, 'total_tokens': 243, 'completion_time': 0.178928137, 'completion_tokens_details': {'reasoning_tokens': 140}, 'prompt_time': 0.005773331, 'prompt_tokens_details': None, 'queue_time': 0.156435759, 'total_time': 0.184701468}, 'model_name

In [24]:
from langchain_core.output_parsers import StrOutputParser #gives the exact output
parser=StrOutputParser()
parser.invoke(response)

'Bonjour, comment vas‑tu\u202f?'

In [25]:
### Using LCEL we can chain the components- model and parser
chain=model|parser
chain.invoke(messages)

'Bonjour, comment vas‑tu\u202f?'

In [26]:
## Prompt templates- 
from langchain_core.prompts import ChatPromptTemplate

generic_template="Translate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)

In [29]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [30]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [31]:
chain=prompt|model|parser

In [32]:
chain.invoke({"language":"French","text":"Hello"})

'Bonjour'

## Langserve helps developers deploy Langchain runnables and chains as a REST API

In [34]:
%pip install fastapi

Note: you may need to restart the kernel to use updated packages.


In [35]:
%pip install uvicorn

Note: you may need to restart the kernel to use updated packages.


In [36]:
import os
print(os.getcwd())

C:\Users\rajsi\NIKITA- GenAI


In [37]:
import os
print(os.path.exists(".env"))
print(os.listdir())

True
['.env', '.git', '.ipynb_checkpoints', 'chatbot_ollama.py', 'Langchain(OpenAI).ipynb', 'Langserve.py', 'LLMappwithLCEL.ipynb', 'OneHotEncoding.ipynb', 'OpenAI_Chatbot.ipynb', 'spamhamdata.csv', 'STOPWORDS, POS, NE.ipynb', 'TOKENIZATION.ipynb']


In [38]:
# to excute- python.Langserve.py
# http://127.0.0.1:8000/docs